# B6. Huấn luyện chính trên Google Colab

Notebook sinh tự động bởi `ml/scripts/make_notebooks.py`. Kế hoạch: `docs/nhan-dang-robot/00-ke-hoach.md`, mục B6.

**Trước khi chạy**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Trong Google Drive phải có thư mục `robotica-ml` chứa `images-raw.zip` (ảnh của bộ dữ liệu).

**Cách chạy**: Runtime → Run all. Ô huấn luyện chạy cả 9 lần (3 kiến trúc × 3 hạt giống), mất nhiều giờ. Mỗi epoch lưu checkpoint vào Drive `robotica-ml/runs/b06/`. **Nếu Colab ngắt giữa chừng**: mở lại notebook, Run all lần nữa, việc huấn luyện chạy tiếp từ epoch đã xong; lần chạy đã xong được bỏ qua.

Ảnh chỉ dùng cho nghiên cứu, không chia sẻ notebook kèm dữ liệu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/robotica-ml'
assert os.path.exists(f'{DRIVE}/images-raw.zip'), 'Không thấy robotica-ml/images-raw.zip trong Google Drive'
os.environ['ROBOTREC_DATA'] = '/content/data'      # ảnh giải nén ra đĩa của máy Colab: đọc nhanh hơn đọc thẳng từ Drive
os.environ['ROBOTREC_RUNS'] = f'{DRIVE}/runs'      # checkpoint và kết quả: ghi vào Drive
os.makedirs(os.environ['ROBOTREC_RUNS'], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!test -d /content/WEB_MO_HINH || git clone --depth 1 https://github.com/ledoankhoi/WEB_MO_HINH.git /content/WEB_MO_HINH
!git -C /content/WEB_MO_HINH pull --ff-only
%cd /content/WEB_MO_HINH
!git log -1 --format='Mã đang dùng: %h %s'
# Colab có sẵn PyTorch, torchvision, numpy, pandas, scipy, matplotlib; chỉ cần thêm timm (ghim đúng phiên bản đã dùng ở máy)
!pip install -q timm==1.0.31
!python -c "import torch, timm; print('torch', torch.__version__, '| timm', timm.__version__, '| GPU', torch.cuda.is_available())"

In [ ]:
# Giải nén ảnh (khoảng 3 GB, vài phút; chỉ làm một lần mỗi phiên Colab)
if not os.path.exists('/content/data/images/raw'):
    os.makedirs('/content/data/images', exist_ok=True)
    !unzip -q {DRIVE}/images-raw.zip -d /content/data/images
!python -c "import pandas as pd, os; s=pd.read_csv('ml/data/manifests/b04_splits.csv.gz'); miss=[f for f in s.file if not os.path.exists('/content/data/images/raw/'+f)]; print('ảnh của bộ chia tập:', len(s), '| thiếu:', len(miss))"

In [ ]:
# Chạy thử 2 epoch trên một phần nhỏ dữ liệu (khoảng 5 phút) để chắc mọi thứ chạy được trên Colab. Có thể bỏ qua ô này.
!python ml/scripts/b06_huan_luyen.py --smoke --arch convnext_tiny

In [ ]:
# Huấn luyện chính: 9 lần chạy, chạy tiếp được nếu bị ngắt
!python ml/scripts/b06_huan_luyen.py --all

In [ ]:
# Tổng hợp báo cáo B06 (chạy được cả khi mới xong một phần) và chép về Drive
!python ml/scripts/b06_bao_cao.py
import shutil
out = f'{DRIVE}/reports/B06'
shutil.rmtree(out, ignore_errors=True)
shutil.copytree('ml/reports/B06', out)
shutil.copy('docs/nhan-dang-robot/bao-cao/B06-huan-luyen.md', out)
from IPython.display import Markdown, display
display(Markdown(open('docs/nhan-dang-robot/bao-cao/B06-huan-luyen.md', encoding='utf-8').read()))